# Fairness-Aware Voting for Regression

## Introduction

This tutorial demonstrates how to use the fairness-aware voting strategy for regression in PipeGenie.

The standard PipeGenie ensemble combines the predictions of the best evolved pipelines using voting weights based only on predictive performance. In this extension, the voting mechanism has been modified so that each estimator receives a weight computed from both its predictive performance and its fairness.

For regression tasks:

- Predictive performance is measured using the **R² score**.
- Fairness is measured as the difference in **Root Mean Squared Error (RMSE)** across sensitive groups.
- The parameter `alpha` controls the trade-off between performance and fairness.

## Load and prepare the Dataset

In [ ]:
import pandas as pd
from pipegenie.fairness import prepare_data

df = pd.read_csv("../datasets/communities.data", header=None)
    
df = df.rename(columns={7: "racepctblack", 127: "ViolentCrimesPerPop"})

y = df["ViolentCrimesPerPop"]

A = prepare_data(df["racepctblack"],n_groups=4,labels=["Q1", "Q2", "Q3", "Q4"])

X = df.drop(columns=["ViolentCrimesPerPop","racepctblack"])
X = pd.get_dummies(X, drop_first=True)

In [2]:
from pipegenie.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.25,random_state=42)
A_train, A_test, _, _ = train_test_split(A,y,test_size=0.25,random_state=42)

## Create the PipeGenie regressor

Enable fairness-aware voting by setting `fair_voting=True`.

The parameter `alpha` determines the balance between predictive performance and fairness.

- `alpha = 1.0` uses only predictive performance.
- `alpha = 0.0` uses only fairness.
- Intermediate values balance both objectives.


In [ ]:
grammar_file = "../../grammars/sample-grammar-regression.xml"

from pipegenie.regression import PipegenieRegressor

model = PipegenieRegressor(
grammar=grammar_file,
generations=5,
pop_size=50,
elite_size=5,
maximization=False,
n_jobs=5,
seed=9,
outdir="samples/regression_2",
fair_voting=True,
alpha=0.6,
sensitive_feature=A_train,
)

## Train the model

In [5]:
model.fit(X_train, y_train)

                                       fitness                                            size                                        fitness_elite                                      size_elite              
                 ----------------------------------------------------    --------------------------------------    ----------------------------------------------------    --------------------------------------
gen    nevals    min           max           avg           std           min    max    avg           std           min           max           avg           std           min    max    avg           std       
0      50        0.13507       2.2303e+81    5.5757e+79    3.482e+80     1      6      2.88          1.3363        0.13507       0.13847       0.13645       0.0012363     2      4      2.6           0.8       
1      34        0.13507       0.21485       0.16165       0.022733      1      6      2.88          1.194         0.13507       0.13722       0.1358        0.0

## Predict

In [ ]:
y_pred = model.predict(X_test)

## Evaluate Performance

In [ ]:
from pipegenie.fairness import fairness_report
from pipegenie.metrics import root_mean_squared_error

# =====================================================
# FAIRNESS REPORT
# =====================================================

report = fairness_report(
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=A_test,
    metric=root_mean_squared_error
)

print("\n==============================")
print("OVERALL PERFORMANCE")
print("==============================")

print(report["overall"])

print("\n==============================")
print("PERFORMANCE BY GROUP")
print("==============================")

print(report["by_group"])

## Evaluate Fairness

In [ ]:
# =====================================================
# STANDARD METRICS
# =====================================================

print("\n==============================")
print("ROOT MEAN SQUARED ERROR")
print("==============================")

print(
    root_mean_squared_error(
        y_test,
        y_pred
    )
)

print("\n==============================")
print("MODEL SCORE (R²)")
print("==============================")

print(
    model.score(
        X_test,
        y_test
    )
)

## Conclusion

This tutorial demonstrated the fairness-aware voting strategy integrated into PipeGenie for regression.

Instead of assigning voting weights solely according to predictive performance, the ensemble combines predictive performance and fairness into a single voting weight. This allows users to explicitly control the trade-off between accuracy and fairness through the `alpha` parameter while keeping the original PipeGenie workflow unchanged.